# Laboratorio 1, Modelo 1: regresión lineal para predecir la temperatura máxima del día siguiente

**Caso:** AlpesPlanck, estación meteorológica de Jena.
**Integrantes:** Juan Camilo Panadero, Jose Luis Parra.

Desarrollamos el ciclo completo de aprendizaje de máquina para el **Modelo 1**: exploración,
calidad de datos, preparación, entrenamiento, evaluación, importancia de variables, verificación
de supuestos y generación de predicciones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro

RANDOM_STATE = 42  # semilla fija, para reproducibilidad
sns.set_theme(style="whitegrid")

## 1. Formalización del problema

$\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$, con $\mathbf{x}_i \in \mathbb{R}^d$, $y_i \in \mathcal{Y}$.

| Elemento | Definición en este problema |
|---|---|
| $\mathbf{x}_i$ | Variables meteorológicas registradas el día $i$ (presión, humedad, viento, ráfaga, fecha) |
| $y_i$ | `temp_max_manana`: temperatura máxima **del día siguiente** (°C), con $\mathcal{Y}=\mathbb{R}$ (**regresión**) |
| $n$ | ≈2500 días de registro (2009–2015), antes de limpiar |
| $\mathcal{H}$ | funciones lineales $f(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + b$ |
| $\ell(y,\hat y)$ | error cuadrático $(y-\hat y)^2$ |
| Métrica pedida | RMSE (y de apoyo, MAE y $R^2$) |

Es importante notar que `y` es la temperatura del **día siguiente**, no la del mismo día: el modelo
usa las condiciones de hoy para anticipar el extremo de mañana, que es justamente lo que
AlpesPlanck necesita para diseñar alertas tempranas (incendios forestales, El Niño / La Niña).

## 2. Carga de datos

In [ ]:
train_raw = pd.read_csv("data/Datos Lab 1.csv")
test_raw = pd.read_csv("data/Datos Test Lab 1.csv")

print("Entrenamiento (etiquetado):", train_raw.shape)
print("Prueba (sin etiquetar):    ", test_raw.shape)
train_raw.head()

## 3. Exploración de los datos

### 3.1 Estructura y tipos

In [ ]:
train_raw.info()

**Lectura de `info()`:**

- `fecha` es `object`: aún no se puede ordenar ni usar `.dt`, hay que convertirla a `datetime64`.
- `anio` y `dia_del_anio` llegan como `float64` aunque son enteros por definición.
- `estacion_anio`, `mes` y `sector_viento` son `object`: son variables **cualitativas nominales**
  (no hay un orden natural entre "invierno" y "verano" como concepto, aunque sí hay un ciclo anual).
- Todas las demás columnas numéricas (presión, humedad, viento, ráfaga) son **cuantitativas
  continuas**.
- `registros_del_dia` es cuantitativa discreta y no es una variable meteorológica: cuenta cuántas
  mediciones de 10 minutos se resumieron ese día (un día completo tiene 144). Es más un indicador
  de calidad del propio registro que un predictor climático, pero la mantenemos como predictor
  porque un día con pocos registros puede llevar promedios menos confiables.

In [ ]:
train_raw.describe(include="all").T

### 3.2 Diccionario de datos

Se revisó `data/Diccionario de datos.xlsx` antes de tocar cualquier columna. Resumen:

| Variable | Descripción | Unidad |
|---|---|---|
| `presion_*` | Presión atmosférica media/mín/máx/desv. del día | mbar |
| `humedad_*` | Humedad relativa media/mín/máx/desv. del día | % |
| `viento_*` | Velocidad del viento media/mín/máx/desv. del día | m/s |
| `rafaga_*` | Ráfagas máximas media/mín/máx/desv. del día | m/s |
| `viento_norte`, `viento_este` | Componentes norte-sur y este-oeste del viento medio | m/s |
| `direccion_viento` | Dirección media del viento | grados 0–360 |
| `registros_del_dia` | Número de mediciones de 10 min resumidas (máx. 144) | conteo |
| `anio`, `dia_del_anio`, `mes`, `estacion_anio` | Componentes de la fecha | N/A |
| `sector_viento` | Sector de la rosa de los vientos (N, NE, E, SE, S, SO, O, NO) | categórica |
| **`temp_max_manana`** | **Variable objetivo**: temperatura máxima del día **siguiente** | °C |

### 3.3 Calidad de datos

Diagnosticamos las cuatro dimensiones de calidad de datos: **completitud, unicidad, consistencia y
validez**. En esta sección solo se *detecta y documenta*; la corrección se hace en la Sección 4.

#### 3.3.1 Completitud

In [ ]:
faltantes = (train_raw.isnull().sum() / len(train_raw)).sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]
faltantes.to_frame("proporcion_faltante")

Casi todas las columnas tienen entre 2 % y 3.5 % de valores faltantes, incluyendo la variable
objetivo `temp_max_manana` (95 filas, 3.7 %). El patrón está disperso entre columnas y no
concentrado en unas pocas filas (lo verificamos con `isnull().sum(axis=1)` más abajo), lo cual es
compatible con un mecanismo **MCAR**: no hay evidencia de que la ausencia dependa de otra variable
observada. Con una proporción tan baja (< 5 %), el criterio adecuado es imputar con estrategias
simples (mediana/moda). En el caso de `temp_max_manana`, como es la etiqueta, simplemente no
podemos entrenar con esas filas y se eliminan.

In [ ]:
nulos_por_fila = train_raw.isnull().sum(axis=1)
print("Filas con 0 nulos:", (nulos_por_fila == 0).sum())
print("Filas con 1 nulo:", (nulos_por_fila == 1).sum())
print("Filas con 2+ nulos:", (nulos_por_fila >= 2).sum())

#### 3.3.2 Unicidad

In [ ]:
print("Filas exactamente duplicadas:", train_raw.duplicated().sum())

con_fecha_valida = train_raw["fecha"].notna()
dup_fecha = train_raw[con_fecha_valida & train_raw["fecha"].duplicated(keep=False)].sort_values("fecha")
print("Filas con 'fecha' repetida (excluyendo las 72 filas sin fecha):", len(dup_fecha))
dup_fecha[["fecha", "presion_media", "humedad_media", "temp_max_manana"]].head(10)

Hay 4 filas idénticas en todas las columnas (duplicados exactos, se eliminan con
`drop_duplicates()`) y otras filas con la misma `fecha` pero valores ligeramente distintos, es
decir, **duplicados lógicos**. Una estación meteorológica solo puede producir un resumen diario por
fecha, así que estas filas adicionales son captura duplicada y no información nueva. Se conserva
una sola fila por fecha.

#### 3.3.3 Consistencia

In [ ]:
for col in ["estacion_anio", "mes", "sector_viento"]:
    valores = sorted(train_raw[col].dropna().unique().tolist())
    print(f"{col} ({len(valores)} variantes distintas):")
    print(valores)
    print()

Las tres variables categóricas están escritas con decenas de variantes del mismo concepto: mayúsculas
y minúsculas, español e inglés, abreviaturas, errores de tecleo ("Inverano", "berano", "invernio",
"verno") y tildes inconsistentes. Es un problema de **formato**, no de contenido: el dato es
correcto, solo está codificado de más de una manera.

En vez de construir un diccionario de reemplazo para 61 variantes de mes y 28 de estación, con el
riesgo de que alguna quede sin mapear, aprovechamos que `fecha` sí trae la información
completa y exacta: **recalculamos `anio`, `dia_del_anio`, `mes` y `estacion_anio` a partir de la
fecha ya convertida a `datetime64`** (con `.dt.year`, `.dt.dayofyear`, `.dt.month`). Esto además
corrige de un solo golpe filas donde el texto contradice la fecha (por ejemplo, una fila con
`fecha=2012-10-04` pero
`mes='febrero'`, que detectamos en la Sección 3.3.4). El único texto categórico que sí normalizamos
con un diccionario de mapeo es `sector_viento`, porque no se puede derivar de otra columna.

#### 3.3.4 Validez

In [ ]:
print("presion_media fuera de rango fisico (>2000 mbar):")
print(train_raw.loc[train_raw["presion_media"] > 2000, ["fecha", "presion_media", "presion_min", "presion_max"]])

`presion_min` y `presion_max` de esas mismas filas están en el rango normal (~970–1010 mbar), y
`presion_media / 10` cae justo entre esos dos límites. El mecanismo es un **error de escala**
(se perdió un punto decimal al capturar el dato). Se corrige dividiendo por 10 solo esas filas.

In [ ]:
print("humedad fuera de [0, 100] %:")
for c in ["humedad_media", "humedad_min", "humedad_max"]:
    n = ((train_raw[c] < 0) | (train_raw[c] > 100)).sum()
    print(f"  {c}: {n} filas")

print()
print("rafaga_min con valor centinela -9999 (sensor caido):", (train_raw["rafaga_min"] == -9999).sum())
print("rafaga_desv negativa (signo invertido):", (train_raw["rafaga_desv"] < 0).sum())
print("registros_del_dia > 144 (maximo fisico posible):", (train_raw["registros_del_dia"] > 144).sum())

- **Humedad relativa** es un porcentaje: por definición no puede ser negativa ni superar 100 %. Los
  pocos valores fuera de rango (apenas por encima de 100, ruido de instrumento) se recortan a
  `[0, 100]` con `Series.clip`.
- **`rafaga_min == -9999`** es un valor centinela clásico para "sensor caído / dato no disponible".
  Cuando aparece, también se corrompen `rafaga_media` y `rafaga_desv` de esa misma fila (fallo del
  mismo sensor), así que las tres se marcan como faltantes (`NaN`) para que las impute el pipeline,
  en vez de inventar un número.
- **`rafaga_desv` negativa** no tiene sentido físico (una desviación estándar siempre es ≥ 0): es un
  error de signo, se corrige con `abs()`.
- **`registros_del_dia` > 144**: un día tiene 144 intervalos de 10 minutos como máximo; valores por
  encima son un error de conteo y se recortan a 144.

In [ ]:
print("temp_max_manana: valores por encima de lo fisicamente plausible para Jena (>35 C):")
sospechosos = train_raw.loc[train_raw["temp_max_manana"] > 35,
                             ["fecha", "mes", "estacion_anio", "temp_max_manana"]].copy()
sospechosos["si_fueran_Fahrenheit"] = (sospechosos["temp_max_manana"] - 32) * 5 / 9
sospechosos

Estas ~15 filas tienen temperaturas de 35–63 °C **en meses de invierno y otoño**, algo
imposible en Jena. Si se interpretan como grados **Fahrenheit** en vez de Celsius y se convierten
con $(F-32)\times 5/9$, el resultado cae exactamente en el rango plausible (2 °C a 17 °C) para esas
fechas. El mecanismo es una mezcla de unidades de temperatura en un subconjunto de registros, y se
corrige con la fórmula de conversión.

### 3.4 Visualización

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_raw["temp_max_manana"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("temp_max_manana (cruda)\nnótese la cola por errores en °F")
axes[0].set_xlabel("°C")

sns.boxplot(x=train_raw["presion_media"], ax=axes[1])
axes[1].set_title("presion_media (cruda)\nnótese el atípico por error de escala")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 9))
corr = train_raw.select_dtypes("number").corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de correlación de variables numéricas (datos crudos)")
plt.tight_layout()
plt.show()

Ya en esta vista cruda se insinúan bloques de variables muy correlacionadas entre sí (las tres
métricas de presión entre ellas, las de humedad entre ellas, viento con ráfaga). Retomamos esto de
forma cuantitativa con el VIF en la Sección 9, porque **multicolinealidad no impide predecir, pero sí
compromete la interpretación de los coeficientes individuales**.

## 4. Preparación de los datos: estrategia del Modelo 1

**Resumen de la estrategia de preparación de este modelo:** correcciones de validez basadas en
reglas de negocio con evidencia clara (Sección 3.3.4), recálculo de las variables de fecha a partir
de `fecha`, normalización de texto para `sector_viento`, eliminación de duplicados, imputación
simple (mediana/moda) dentro del `Pipeline` porque la proporción de faltantes es baja, y
estandarización + codificación *one-hot* para que el modelo lineal pueda usar las variables
categóricas y los coeficientes numéricos sean comparables entre sí. No se transforma ninguna
variable (sin logaritmos ni polinomios) y no se eliminan variables por colinealidad.

Escribimos la limpieza como una función para aplicarla **exactamente igual** sobre el conjunto de
entrenamiento y sobre el conjunto de prueba sin etiquetar (evita que se nos olvide un paso al
generar las predicciones finales de la Sección 10).

In [ ]:
MAPA_SECTOR = {
    "E": "E", "ESTE": "E", "EAST": "E",
    "N": "N", "NORTE": "N", "NORTH": "N",
    "NE": "NE", "NORESTE": "NE", "NORTHEAST": "NE",
    "NO": "NO", "NOROESTE": "NO", "NORTHWEST": "NO",
    "O": "O", "OESTE": "O", "WEST": "O",
    "S": "S", "SUR": "S", "SOUTH": "S",
    "SE": "SE", "SURESTE": "SE", "SOUTHEAST": "SE",
    "SO": "SO", "SUROESTE": "SO", "SOUTHWEST": "SO",
}

MAPA_ESTACION = {
    12: "invierno", 1: "invierno", 2: "invierno",
    3: "primavera", 4: "primavera", 5: "primavera",
    6: "verano", 7: "verano", 8: "verano",
    9: "otono", 10: "otono", 11: "otono",
}


def limpiar(df, formato_fecha):
    # Corrige completitud/unicidad/consistencia/validez detectadas en la Sección 3.
    # Se aplica igual sobre entrenamiento y sobre el conjunto de prueba sin etiquetar.
    df = df.copy()

    # Consistencia: normalizar sector_viento a los 8 sectores canonicos
    df["sector_viento"] = (
        df["sector_viento"].astype(str).str.strip().str.upper()
        .replace("NAN", np.nan)
        .map(MAPA_SECTOR)
    )

    # Consistencia + validez: recalcular las variables de fecha desde 'fecha'
    df["fecha"] = pd.to_datetime(df["fecha"], format=formato_fecha, errors="coerce")
    df["anio"] = df["fecha"].dt.year
    df["dia_del_anio"] = df["fecha"].dt.dayofyear
    df["mes"] = df["fecha"].dt.month
    df["estacion_anio"] = df["mes"].map(MAPA_ESTACION)

    # Validez: presion_media con error de escala (falta un punto decimal)
    fuera_de_rango = df["presion_media"] > 2000
    df.loc[fuera_de_rango, "presion_media"] = df.loc[fuera_de_rango, "presion_media"] / 10

    # Validez: humedad fuera del rango fisico [0, 100] %
    for c in ["humedad_media", "humedad_min", "humedad_max"]:
        df[c] = df[c].clip(lower=0, upper=100)

    # Validez: valor centinela -9999 en rafaga_min (fallo de sensor)
    centinela = df["rafaga_min"] == -9999
    df.loc[centinela, ["rafaga_min", "rafaga_media", "rafaga_desv"]] = np.nan

    # Validez: rafaga_desv con signo invertido
    df["rafaga_desv"] = df["rafaga_desv"].abs()

    # Validez: registros_del_dia no puede superar 144 (24h x 6 registros/hora)
    df["registros_del_dia"] = df["registros_del_dia"].clip(upper=144)

    return df


# El formato de fecha difiere entre los dos archivos (se verificó al explorar ambos):
# train: 'YYYY-MM-DD'   test: 'DD.MM.YYYY'
train = limpiar(train_raw, formato_fecha="%Y-%m-%d")
test = limpiar(test_raw, formato_fecha="%d.%m.%Y")

train.head()

In [ ]:
# Validez: temp_max_manana en grados Fahrenheit en vez de Celsius (solo aplica al target)
mascara_fahrenheit = train["temp_max_manana"] > 35
print("Filas de la variable objetivo corregidas de F a C:", mascara_fahrenheit.sum())

train.loc[mascara_fahrenheit, "temp_max_manana"] = (
    (train.loc[mascara_fahrenheit, "temp_max_manana"] - 32) * 5 / 9
)

sns.histplot(train["temp_max_manana"], bins=50, kde=True)
plt.title("temp_max_manana ya corregida")
plt.xlabel("°C")
plt.show()

In [ ]:
# Unicidad: eliminar duplicados exactos
antes = len(train)
train = train.drop_duplicates()
print(f"Duplicados exactos eliminados: {antes - len(train)}")

# Unicidad: eliminar duplicados logicos por fecha (conservamos el primer registro)
# Excluimos del criterio de deduplicacion las filas sin fecha valida, porque pandas trata
# NaN == NaN como duplicado y borraria filas distintas que solo comparten el dato faltante.
con_fecha = train["fecha"].notna()
antes = con_fecha.sum()
train_con_fecha = (
    train[con_fecha].sort_values("fecha").drop_duplicates(subset="fecha", keep="first")
)
train_sin_fecha = train[~con_fecha]
print(f"Duplicados logicos por fecha eliminados: {antes - len(train_con_fecha)}")

train = pd.concat([train_con_fecha, train_sin_fecha], ignore_index=True)
print("Filas tras deduplicar:", len(train))

In [ ]:
# Completitud: no se puede entrenar con filas sin variable objetivo
antes = len(train)
train = train.dropna(subset=["temp_max_manana"])
print(f"Filas sin 'temp_max_manana' eliminadas: {antes - len(train)} "
      f"({(antes - len(train)) / antes:.2%})")
print("Filas finales de entrenamiento:", len(train))

El resto de valores faltantes (~3 % por columna, mecanismo MCAR según la Sección 3.3.1) **no se
imputa aquí**: se deja para el `Pipeline` de la Sección 6, que ajusta la mediana/moda **solo con el
conjunto de entrenamiento** tras la partición de la Sección 5. Imputar antes de partir sería fuga de
datos.

In [ ]:
train.isnull().sum().to_frame("faltantes_restantes").query("faltantes_restantes > 0")

### 4.1 Definición de $X$, $y$ y separación de columnas

Excluimos `fecha` como predictor (ya extrajimos de ella `anio`, `dia_del_anio`, `mes` y
`estacion_anio`, que sí entran al modelo) para no duplicar la misma información en dos formas
distintas.

In [ ]:
cols_num = [
    "presion_media", "presion_min", "presion_max", "presion_desv",
    "humedad_media", "humedad_min", "humedad_max", "humedad_desv",
    "viento_media", "viento_min", "viento_max", "viento_desv",
    "rafaga_media", "rafaga_min", "rafaga_max", "rafaga_desv",
    "viento_norte", "viento_este", "direccion_viento",
    "registros_del_dia", "anio", "dia_del_anio", "mes",
]
cols_cat = ["estacion_anio", "sector_viento"]

X = train[cols_num + cols_cat]
y = train["temp_max_manana"]
X_no_etiquetado = test[cols_num + cols_cat]  # para la Sección 10

print("X:", X.shape, " y:", y.shape, " X_no_etiquetado:", X_no_etiquetado.shape)

## 5. Partición entrenamiento / prueba

Usamos `random_state=42` y `test_size=0.25`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)
print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

## 6. Modelo 1: `Pipeline` + regresión lineal

`ColumnTransformer` aplica dos ramas distintas:

- **Numéricas:** imputación por la mediana (robusta a los atípicos que no corregimos, como
  `viento_desv`) y estandarización (`StandardScaler`), necesaria para que los coeficientes sean
  comparables entre variables en la Sección 8.
- **Categóricas:** imputación por la moda y codificación *one-hot* con `drop='first'` (evita la
  trampa de la variable ficticia) y `handle_unknown='ignore'` (por si el conjunto de prueba
  trajera una categoría no vista, aunque en este caso ya verificamos que no).

Al envolver todo en un `Pipeline`, `fit` solo ve `X_train` y `predict`/`transform` reutilizan esos
mismos parámetros sobre `X_test` y sobre el conjunto sin etiquetar: así se evita la fuga de datos.

In [ ]:
rama_numerica = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

rama_categorica = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocesador = ColumnTransformer([
    ("num", rama_numerica, cols_num),
    ("cat", rama_categorica, cols_cat),
])

modelo_1 = Pipeline([
    ("preparacion", preprocesador),
    ("regresion", LinearRegression()),
])

modelo_1.fit(X_train, y_train)
print("Modelo 1 entrenado.")

## 7. Evaluación cuantitativa

In [ ]:
def evaluar(y_real, y_pred, nombre):
    rmse = root_mean_squared_error(y_real, y_pred)
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    print(f"{nombre:14s} RMSE={rmse:6.3f} °C   MAE={mae:6.3f} °C   R²={r2:6.4f}")
    return {"conjunto": nombre, "RMSE": rmse, "MAE": mae, "R2": r2}


pred_train = modelo_1.predict(X_train)
pred_test = modelo_1.predict(X_test)

m_train = evaluar(y_train, pred_train, "Entrenamiento")
m_test = evaluar(y_test, pred_test, "Prueba")

resultados_modelo1 = pd.DataFrame([m_train, m_test]).set_index("conjunto")
resultados_modelo1

**Lectura.** El RMSE de entrenamiento y de prueba están muy cerca entre sí (no hay señal de
sobreajuste severo). Con `temp_max_manana` moviéndose entre −15 °C y 35 °C aproximadamente
(recorrido de ~50 °C), un RMSE de esta magnitud implica que el modelo explica una fracción
razonable pero no total de la variabilidad, coherente con el $R^2$: el clima del día siguiente
depende de dinámicas atmosféricas que no están completamente capturadas por los promedios diarios
de hoy (frentes que llegan mañana, por ejemplo), así que un techo de desempeño lejos de 1.0 es
esperable y no un error de implementación.

### Tabla comparativa

`#TODO`

In [ ]:
tabla_comparativa = pd.DataFrame([
    {
        "modelo": "Modelo 1 (imputación mediana/moda + estandarización + one-hot, sin winsorizar)",
        "RMSE_test": m_test["RMSE"],
        "MAE_test": m_test["MAE"],
        "R2_test": m_test["R2"],
    },
    # TODO
])
tabla_comparativa

## 8. Importancia de variables

Como las variables numéricas quedaron estandarizadas dentro del pipeline, sus coeficientes son
directamente comparables entre sí: cada uno representa el cambio esperado en `temp_max_manana`
(°C) ante un aumento de **una desviación estándar** en esa variable, manteniendo las demás fijas.
Los coeficientes de las variables categóricas (*dummies* 0/1) se interpretan distinto: son el
efecto de pertenecer a esa categoría frente a la categoría de referencia que `drop='first'` dejó
fuera.

In [ ]:
nombres_variables = modelo_1.named_steps["preparacion"].get_feature_names_out()
coeficientes = modelo_1.named_steps["regresion"].coef_

importancia = pd.DataFrame({
    "variable": nombres_variables,
    "coeficiente": coeficientes,
})
importancia["abs_coeficiente"] = importancia["coeficiente"].abs()
importancia = importancia.sort_values("abs_coeficiente", ascending=False).drop(columns="abs_coeficiente")
importancia = importancia.reset_index(drop=True)

print("Intercepto:", modelo_1.named_steps["regresion"].intercept_)
importancia

In [ ]:
top10 = importancia.head(10).iloc[::-1]
plt.figure(figsize=(8, 5))
colores = ["#2E86AB" if v > 0 else "#C7402D" for v in top10["coeficiente"]]
plt.barh(top10["variable"], top10["coeficiente"], color=colores)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coeficiente estandarizado (°C)")
plt.title("Modelo 1: 10 variables con mayor importancia")
plt.tight_layout()
plt.show()

**Lectura para AlpesPlanck.** Las variables de mayor peso son las categorías de `estacion_anio`
(el efecto estacional domina, como es de esperar en un fenómeno con ciclo anual tan marcado), la
`dia_del_anio` (refuerzo del mismo ciclo dentro de la estación), el `sector_viento` (el viento del
noroeste se asocia con temperaturas más bajas al día siguiente, consistente con masas de aire
frío desde el Atlántico norte) y `humedad_desv` (mayor variabilidad de humedad durante el día
anticipa un cambio de temperatura más marcado para el día siguiente). Esto es información
accionable: confirma que la estación del año y la dirección del viento son las señales más fuertes
para anticipar extremos térmicos, más que la presión o la velocidad del viento en sí.

## 9. Verificación de los supuestos del modelo lineal

Usamos `statsmodels` sobre la matriz ya transformada por el pipeline (mismo conjunto de
entrenamiento) para poder inspeccionar residuales.

In [ ]:
X_train_transformado = modelo_1.named_steps["preparacion"].transform(X_train)
if hasattr(X_train_transformado, "toarray"):
    X_train_transformado = X_train_transformado.toarray()

X_train_con_constante = sm.add_constant(X_train_transformado)
modelo_ols = sm.OLS(y_train.values, X_train_con_constante).fit()

print("R² (statsmodels, debe coincidir con sklearn):", round(modelo_ols.rsquared, 4))
residuales = modelo_ols.resid

### 9.1 Multicolinealidad (VIF)

Calculado sobre las variables numéricas **sin estandarizar** (el VIF no depende de la escala).

In [ ]:
X_num_train = X_train[cols_num].fillna(X_train[cols_num].median())
X_num_con_constante = sm.add_constant(X_num_train)

vif = pd.DataFrame({
    "variable": X_num_con_constante.columns,
    "VIF": [variance_inflation_factor(X_num_con_constante.values, i)
            for i in range(X_num_con_constante.shape[1])],
})
vif = vif[vif["variable"] != "const"].sort_values("VIF", ascending=False).reset_index(drop=True)
vif

`dia_del_anio` y `mes` tienen un VIF altísimo (ambas describen la misma posición dentro del año, una
como número continuo y otra como entero 1–12: son casi combinación lineal exacta una de otra).
`viento_norte` y `viento_desv` también quedan por encima del umbral convencional de 10, porque están
mecánicamente ligadas a `direccion_viento` y a `viento_media`. Esto **no invalida las predicciones**
del modelo (para eso el VIF no importa), pero sí advierte que los coeficientes individuales de esas
variables en la Sección 8 no deben leerse de forma aislada: parte de su efecto se está repartiendo
entre variables redundantes.

### 9.2 Independencia de los residuales (Durbin–Watson)

In [ ]:
dw_particion_aleatoria = durbin_watson(residuales)
print("Durbin-Watson (orden de la partición aleatoria):", round(dw_particion_aleatoria, 3))

In [ ]:
# Los datos son una serie diaria: el orden temporal real importa para este supuesto,
# aunque la partición para entrenar y evaluar se haya hecho de forma aleatoria.
fechas_train = train.loc[X_train.index, "fecha"]
orden_temporal = fechas_train.sort_values().index
residuales_por_fecha = pd.Series(residuales, index=X_train.index).loc[orden_temporal]

dw_orden_temporal = durbin_watson(residuales_por_fecha.values)
print("Durbin-Watson (ordenado por fecha real):", round(dw_orden_temporal, 3))

Con el orden que deja la partición aleatoria, el estadístico da cerca de 2 (sugiere independencia).
Pero si se ordenan esos mismos residuales por la fecha real, el valor cae a menos de 1: hay
**autocorrelación positiva fuerte**. Es exactamente lo que se espera en una serie de tiempo diaria:
si el modelo sobreestima la temperatura de un día, es probable que también sobreestime la del día
siguiente, porque las condiciones atmosféricas no cambian de un día para otro. La partición
aleatoria esconde este problema porque mezcla días de todo el rango temporal en ambos conjuntos;
lo dejamos documentado porque es central para la pregunta de sesgos de la Sección 11.

### 9.3 Homocedasticidad (Breusch–Pagan)

In [ ]:
lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(residuales, X_train_con_constante)
print(f"Breusch-Pagan: estadístico={lm_stat:.2f}  p-valor={lm_pvalue:.2e}")

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(modelo_ols.fittedvalues, residuales, alpha=0.3, s=12)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Valores ajustados (°C)")
plt.ylabel("Residual (°C)")
plt.title("Residuales vs. valores ajustados")
plt.tight_layout()
plt.show()

El p-valor es muy inferior a 0.05: se rechaza la homocedasticidad. El gráfico de residuales contra
ajustados confirma que la dispersión no es constante: hay más variabilidad en los extremos fríos y
cálidos que en la zona templada central, algo razonable en un fenómeno físico con eventos extremos
más difíciles de anticipar con un modelo lineal simple.

### 9.4 Normalidad de los residuales

In [ ]:
muestra = residuales if len(residuales) <= 5000 else np.random.choice(residuales, 5000, replace=False)
estadistico, p_valor = shapiro(muestra)
print(f"Shapiro-Wilk: W={estadistico:.4f}  p-valor={p_valor:.2e}")

In [ ]:
fig = sm.qqplot(residuales, line="45", fit=True)
plt.title("Gráfico Q-Q de los residuales")
plt.tight_layout()
plt.show()

El p-valor del test también rechaza la normalidad, pero con más de 1800 observaciones el test es
muy sensible a desviaciones pequeñas y prácticamente cualquier apartamiento leve de la normal
alcanza a rechazarse. El gráfico Q-Q es el criterio que realmente importa aquí: los puntos siguen
la diagonal en la parte central y se separan levemente en las colas, coherente con lo observado en
la Sección 9.3 (mayor error en temperaturas extremas).

### 9.5 Síntesis de los cinco supuestos

| Supuesto | Resultado | Riesgo para la interpretación |
|---|---|---|
| Linealidad | Coherente con el gráfico de residuales, sin patrón curvo marcado | Bajo |
| Independencia | Se cumple bajo la partición aleatoria; **se viola** al ordenar por fecha real | Medio-alto para pronóstico secuencial |
| Homocedasticidad | Se rechaza (Breusch–Pagan); mayor error en temperaturas extremas | Medio |
| Normalidad | Se rechaza formalmente; el gráfico Q-Q es razonable salvo en las colas | Bajo-medio |
| Sin multicolinealidad | Se viola para `mes`/`dia_del_anio` y para variables de viento | Alto para leer coeficientes individuales, bajo para predecir |

Ninguno de estos hallazgos invalida usar el modelo para predecir (para eso solo hace falta que
generalice bien, que es lo que medimos con RMSE/MAE/R² sobre prueba). Sí limitan qué tan literal
puede leerse cada coeficiente de la Sección 8 de forma aislada.

## 10. Generación de predicciones sobre el conjunto de prueba no etiquetado

Se usa el mismo `X_no_etiquetado` preparado en la Sección 4.1 (con la misma función `limpiar` y el
mismo pipeline ajustado únicamente sobre `X_train`).

In [ ]:
predicciones = modelo_1.predict(X_no_etiquetado)
print(pd.Series(predicciones).describe())

In [ ]:
salida = test_raw.copy()
salida["temp_max_manana"] = predicciones

# Se guarda con un nombre distinto al archivo original de entrada (data/Datos Test Lab 1.csv,
# que se deja intacto sin etiquetar) para no confundir el insumo con el resultado. El nombre
# final para la entrega se define una vez se elija el mejor de los dos modelos (Sección 10.1).
ruta_salida = "Datos Test Lab 1 - predicciones Modelo 1.csv"
salida.to_csv(ruta_salida, index=False)
print(f"Archivo exportado: {ruta_salida}")
salida.head()

El archivo conserva exactamente las columnas originales de `data/Datos Test Lab 1.csv` (que
permanece sin modificar) y agrega la columna `temp_max_manana` con la predicción del **Modelo 1**.

### 10.1 Nombre final para la entrega

Cuando se elija el mejor de los dos modelos (Sección 7), el archivo de ese modelo debe copiarse o
renombrarse a `Datos Test Lab 1.csv` antes de subirlo. No se hace ese renombrado automáticamente
aquí para no pisar por accidente el archivo original de entrada que vive en `data/`.

## 11. Preguntas de análisis de resultados

**¿Cuál fue el valor de los diferentes coeficientes obtenidos en el mejor modelo?**
`#TODO`

**A partir de la tabla comparativa, ¿cuál modelo ofrece el mejor rendimiento sobre el conjunto de
prueba? ¿Qué interpretación puedes darle a los valores obtenidos sobre las métricas de rendimiento?**
`#TODO`

**¿Cuáles variables fueron seleccionadas con el modelo seleccionado? A partir de estas, ¿qué
interpretación de cara al problema puedes dar?**
El Modelo 1 no hace selección de variables (usa todas las disponibles), pero la tabla de importancia
de la Sección 8 muestra que la señal más fuerte es estacional (estación del año, día del año) y de
dirección del viento. Para AlpesPlanck esto sugiere que las alertas más confiables se pueden anclar
a la época del año y al régimen de vientos dominante, más que a la presión atmosférica puntual del
día. Es información útil para priorizar qué sensores mantener con mayor precisión.

**A partir del contexto y los datos compartidos, ¿cómo representar la regresión lineal de forma
matemática? Indique el método utilizado y el proceso para resolverlo.**
Con $\mathbf{X}\in\mathbb{R}^{n\times(d+1)}$ (incluyendo la columna de unos del intercepto) y
$\mathbf{y}\in\mathbb{R}^n$, el modelo es $\hat y = \mathbf{X}\mathbf{w}$ y se ajusta minimizando el
riesgo empírico de mínimos cuadrados $J(\mathbf{w}) = \frac{1}{2n}\lVert \mathbf{y}-\mathbf{X}\mathbf{w}\rVert_2^2$.
`LinearRegression` de `scikit-learn` no resuelve las ecuaciones normales
$\hat{\mathbf{w}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ directamente (sería
costoso e inestable si hay columnas casi colineales, como se vio en la Sección 9.1); usa una
descomposición en valores singulares (SVD), numéricamente más estable.

**En el ciclo de machine learning, ¿qué tipos de sesgo podrían afectar los resultados? Describa dos
tipos de sesgo.**
1. *Sesgo de muestra*: la estación de Jena registra un único punto geográfico entre 2009 y 2015. Un
   modelo entrenado ahí no necesariamente generaliza a otra ciudad o a un clima futuro distinto
   (más cálido, con eventos más extremos) al que se entrenó, que es justamente el escenario que
   preocupa a AlpesPlanck (El Niño/La Niña, cambio climático).
2. *Sesgo de evaluación por ignorar la estructura temporal*: la partición aleatoria
   (`train_test_split` con `random_state=42`) mezcla días de todo el rango de años en
   entrenamiento y en prueba. La Sección 9.2 mostró que los residuales están fuertemente
   autocorrelacionados en el tiempo real; una partición aleatoria puede sobreestimar qué tan bien
   generalizaría el modelo a pronosticar días *futuros* nunca vistos (que es el uso real que le dará
   AlpesPlanck), porque dejó "fugarse" información de días cercanos en el tiempo entre los dos
   conjuntos. Una partición cronológica (entrenar con años tempranos, evaluar con los últimos)
   sería una prueba más honesta de ese caso de uso real.